# A — Corpus to triples (GPU)

Everything that needs a GPU, in one place: download → normalise → extract triples.
Downstream work (entity resolution, graph, verifier, evaluation) lives in notebook B and
runs on CPU against this notebook's output.

**Every stage caches.** Each writes its output and skips if the file already exists, and
extraction checkpoints per chunk — a session that dies at chunk 900 resumes at 900, it
does not start over.

**Settings:** GPU `NvidiaTeslaT4` (a P100 cannot run this image's PyTorch), Internet on.

Replaces notebooks 01, 02 and 03.

In [ ]:
# Kaggle's image ships no bitsandbytes. Must run before any transformers import; if this
# session already failed on that import, restart the session first.
!pip install -q -U "bitsandbytes>=0.46.1" accelerate

import importlib.util
assert importlib.util.find_spec("bitsandbytes"), "bitsandbytes missing — restart the session"
print("bitsandbytes ready")

In [ ]:
import json, glob, re, zipfile, shutil, collections, urllib.request, time, os
from pathlib import Path
import pandas as pd

# ---- config -------------------------------------------------------------
SUBJECT   = "biology_secondary"
MODEL_SIZE = "7b-instruct"    # 14b OOMs on 2xT4: this image ignores max_memory under bnb
CHAPTERS  = [6]      # None = all 14
LIMIT     = 8        # chunks per chapter; None = all. Raise once output looks right.
MAX_NEW   = 1024     # 512 truncated dense chunks mid-JSON
BATCH     = 4
# -------------------------------------------------------------------------

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
# Corpus intermediates go to /tmp: anything under /kaggle/working becomes kernel
# output, and the extracted corpus is thousands of files that bury the results.
SCRATCH = Path("/tmp")
ZIP_PATH   = SCRATCH / "NCTB-SchoolText.zip"
CORPUS     = SCRATCH / "nctb"
CLEAN_PQ   = WORK / "biology_clean.parquet"
TRIPLES_JL = WORK / "triples_raw.jsonl"     # per-chunk checkpoint
TRIPLES_CSV = WORK / "triples.csv"

MENDELEY_URL = "https://data.mendeley.com/public-files/datasets/f3882ccczp/files/54e66048-5d2b-47a0-a489-a0b104119476/file_downloaded"
UA = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0 Safari/537.36"
BENGALI_TOKEN = re.compile(r"[ঀ-৿]+")
print(WORK)

## Stage 1 — corpus

In [ ]:
if CORPUS.exists():
    print("corpus already present, skipping download")
else:
    if not ZIP_PATH.exists():
        # Mendeley's CDN 403s the default Python-urllib User-Agent.
        req = urllib.request.Request(MENDELEY_URL, headers={"User-Agent": UA})
        with urllib.request.urlopen(req, timeout=120) as r, open(ZIP_PATH, "wb") as f:
            shutil.copyfileobj(r, f)
    if not zipfile.is_zipfile(ZIP_PATH):
        ZIP_PATH.unlink()
        raise SystemExit("Downloaded file is not a zip — CDN served an error page.")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(CORPUS)
    print("extracted")

## Stage 2 — normalise

Two repairs. **Chapter titles**: 6 of 14 stored titles are OCR-damaged and they become node
labels, so they're replaced with the printed textbook's wording. **`ক্ত`/`স্ত`**: the OCR
misreads the `ক্ত` conjunct, splitting entities like `রক্ত` across two spellings.

The `স্ত` repair is deliberately conservative — a token is only corrected when its `ক্ত`
form independently appears elsewhere in the corpus. A blanket substitution would wreck
`বস্তু`, `মস্তিষ্ক`, `স্তর`. Replacement is whole-token: as a substring pass, the key `রস্ত`
also matches inside প্র+স্তুত and destroys `প্রস্তুত`.

Known gap: the OCR also produces `ন্ত` for `ক্ত` (`রন্তচাপ`), which this does not catch —
it showed up as a duplicate node in the first extraction run. Left alone for now.

In [ ]:
TITLES = {
    1:  ("জীবন পাঠ", "জীবনপাঠ"),        2:  ("জীবকোষ ও টিস্যু", None),
    3:  ("কোষ বিভাজন", None),            4:  ("জীবনীশক্তি", None),
    5:  ("খাদ্য, পুষ্টি ও পরিপাক", None), 6:  ("জীবে পরিবহন", "জীবে পরিবহণ"),
    7:  ("গ্যাসীয় বিনিময়", None),        8:  ("রেচন প্রক্রিয়া", None),
    9:  ("দৃঢ়তা প্রদান ও চলন", None),   10: ("সমন্বয়", None),
    11: ("জীবের প্রজনন", None),
    12: ("জীবের বংশগতি ও জৈব অভিব্যক্তি", "জীবের বংশগতি ও বিবর্তন"),
    13: ("জীবের পরিবেশ", None),          14: ("জীবপ্রযুক্তি", None),
}
BLOCKLIST = {"ব্যস্ত"}          # real word whose ক্ত-twin ব্যক্ত also exists
EXTRA_FIX = {"সৌরশস্তিকে": "সৌরশক্তিকে", "দৃষ্টিশস্তি": "দৃষ্টিশক্তি",
             "বাকশস্তি": "বাকশক্তি", "ইচ্ছাশস্তির": "ইচ্ছাশক্তির",
             "চিন্তাশস্তি": "চিন্তাশক্তি"}


def bengali_ratio(s):
    letters = [c for c in s if c.isalnum()]
    return sum(1 for c in letters if "ঀ" <= c <= "৿") / len(letters) if letters else 0.0


if CLEAN_PQ.exists():
    clean = pd.read_parquet(CLEAN_PQ)
    print(f"cached: {len(clean)} clean chunks")
else:
    rows = []
    for f in sorted(glob.glob(f"{CORPUS}/classNineTen/processed_chapters_{SUBJECT}/*.jsonl")):
        with open(f, encoding="utf-8") as fh:
            rows += [json.loads(l) for l in fh if l.strip()]
    df = pd.DataFrame(rows)

    counts = collections.Counter()
    for t in df.text:
        counts.update(BENGALI_TOKEN.findall(t))
    # The OCR mangles ক্ত two ways: into স্ত and into ন্ত. Same corroboration rule for
    # both — only correct a token when its ক্ত form independently occurs in the corpus.
    # ন্ত is by far the more damaging of the two (144 occurrences vs 86) because it hits
    # রক্ত and its compounds, and it is also a very common legitimate conjunct, which is
    # why the corroboration test rather than a blanket substitution.
    kta = {w for w in counts if "ক্ত" in w}
    FIXES = {}
    for wrong in ("স্ত", "ন্ত"):
        FIXES.update({w: w.replace(wrong, "ক্ত") for w in counts
                      if wrong in w and w not in BLOCKLIST
                      and w.replace(wrong, "ক্ত") in kta})
    FIXES.update(EXTRA_FIX)

    before = "\n".join(df.text)
    df["text"] = df.text.map(
        lambda t: BENGALI_TOKEN.sub(lambda m: FIXES.get(m.group(), m.group()), t))
    after = "\n".join(df.text)

    tb = collections.Counter(BENGALI_TOKEN.findall(before))
    ta = collections.Counter(BENGALI_TOKEN.findall(after))
    for w in ["বস্তু", "মস্তিষ্ক", "স্তর", "প্রস্তুত", "স্ত্রী",
              "অন্ত", "চিন্তা", "সন্তান", "শান্ত", "যন্ত্র", "তন্ত্র", "বাস্তুতন্ত্র"]:
        assert tb[w] == ta[w], f"REGRESSION: {w} {tb[w]} -> {ta[w]}"

    # Chunk-level filtering misses noise *inside* an otherwise-good chunk; those
    # fragments were surviving into triples as entity names.
    def strip_noise_lines(text):
        keep = [ln for ln in text.split("\n")
                if not ln.strip() or bengali_ratio(ln) >= 0.55 or len(ln.strip()) < 4]
        return "\n".join(keep)

    df["text"] = df.text.map(strip_noise_lines)

    df["chapter_title"] = df.chapter_no.map(lambda c: TITLES[c][0])
    df["chapter_title_alt"] = df.chapter_no.map(lambda c: TITLES[c][1])
    df["n_chars"] = df.text.str.len()
    df["is_junk"] = (df.n_chars < 80) | (df.text.map(bengali_ratio) < 0.5)

    clean = (df[~df.is_junk]
             [["chunk_id", "class", "subject", "chapter_no", "chapter_title",
               "chapter_title_alt", "text", "n_chars"]]
             .sort_values(["chapter_no", "chunk_id"]).reset_index(drop=True))
    clean.to_parquet(CLEAN_PQ, index=False)
    print(f"{len(FIXES)} token repairs; kept {len(clean)}/{len(df)} chunks")

todo = clean if CHAPTERS is None else clean[clean.chapter_no.isin(CHAPTERS)]
if LIMIT:
    todo = todo.groupby("chapter_no", group_keys=False).head(LIMIT)
print(f"queued for extraction: {len(todo)} chunks from chapters {sorted(todo.chapter_no.unique())}")

## Stage 3 — model

In [ ]:
# Must precede CUDA init: reduces fragmentation during sharded 4-bit loading.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Kaggle mounts models at /kaggle/input/models/<owner>/<model>/<framework>/<variation>/<ver>,
# not /kaggle/input/<model>. Match on the variation directory itself, and fail loudly rather
# than falling back to whatever config.json turns up first — a silent fallback previously
# loaded 7b while the notebook claimed to be running 14b.
cfgs = glob.glob(f"/kaggle/input/**/{MODEL_SIZE}/**/config.json", recursive=True)
if not cfgs:
    avail = sorted({p.split("/transformers/")[-1].split("/")[0]
                    for p in glob.glob("/kaggle/input/**/config.json", recursive=True)
                    if "/transformers/" in p})
    raise SystemExit(f"{MODEL_SIZE} not attached. Available variations: {avail or 'none'}")

MODEL_PATH = str(Path(cfgs[0]).parent)
assert f"/{MODEL_SIZE}/" in MODEL_PATH + "/", f"resolved wrong model: {MODEL_PATH}"

tok = AutoTokenizer.from_pretrained(MODEL_PATH)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"

# device_map="auto" alone packed ~14.4 GiB onto one T4 and OOM'd while loading 14B.
# An explicit per-GPU budget forces the shards to balance across both cards; double
# quantisation trims roughly another 0.4 GB off the weights.
n_gpu = torch.cuda.device_count()
max_memory = {i: "12GiB" for i in range(n_gpu)}
max_memory["cpu"] = "8GiB"
print(f"{n_gpu} GPU(s):", [torch.cuda.get_device_name(i) for i in range(n_gpu)])

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    ),
    device_map="auto",
    max_memory=max_memory,
    low_cpu_mem_usage=True,
).eval()
print("loaded:", MODEL_PATH)

## Stage 4 — extraction

Two changes from the first run, both driven by what it actually produced:

- **Relations are now a fixed list.** The unconstrained run gave 16 relations for 36 triples,
  including one-off verb phrases (`প্রকোপ ছড়িয়ে পড়েছে`) that can't be edge types. The list
  below is drawn from what the model chose on its own, so it is a starting vocabulary to
  revise once you've read a larger sample — not a settled schema.
- **Exercises are excluded.** It was extracting activity instructions addressed to the
  student (`তুমি → নাড়িস্পন্দন গণনা করো`) as biology facts.

In [ ]:
RELATIONS = ["সংজ্ঞা", "অংশ", "প্রকার", "কাজ", "অবস্থান",
             "কারণ", "লক্ষণ", "প্রতিরোধ", "পরিমাণ"]

SYSTEM = "তুমি একজন পাঠ্যবই বিশ্লেষক। তুমি শুধুমাত্র বৈধ JSON উত্তর দাও।"

PROMPT = """নিচে বাংলাদেশের নবম-দশম শ্রেণির জীববিজ্ঞান পাঠ্যবইয়ের একটি অংশ দেওয়া হলো।

এই অংশে যেসব তথ্য স্পষ্টভাবে বলা হয়েছে, সেগুলো (subject, relation, object) আকারে বের করো।

relation অবশ্যই নিচের তালিকা থেকে একটি হতে হবে, অন্য কোনো শব্দ ব্যবহার করা যাবে না:
{relations}

নিয়ম:
- শুধু পাঠ্যাংশে যা সরাসরি বলা আছে তাই নাও। নিজে থেকে কিছু যোগ করবে না।
- subject ও object বাংলায়, বইয়ের শব্দ হুবহু রাখবে।
- subject সংক্ষিপ্ত নামপদ হবে (যেমন "উচ্চ রক্তচাপ"), বাক্য নয়, "...এর লক্ষণ" ধরনের নয়।
- **না-বাচক তথ্য বাদ দেবে।** "লক্ষণ প্রকাশ পায় না" ধরনের বাক্য থেকে কোনো triple বানাবে না।
  object কখনো "না", "নেই", "নয়" হবে না।
- ছাত্রছাত্রীর উদ্দেশে দেওয়া কাজ বা নির্দেশ ('করো', 'দেখো', 'গণনা করো') বাদ দেবে।
- "তুমি", "তোমরা", "আমরা" কখনো subject হবে না।
- শিরোনাম, পৃষ্ঠা নম্বর বা অর্থহীন অক্ষর থেকে triple বানাবে না।
- সংখ্যা বা পরিমাপ থাকলে relation হবে "পরিমাণ", object-এ একক সহ লিখবে।
- কোনো তথ্য না থাকলে খালি তালিকা দেবে।

উদাহরণ ১
পাঠ্যাংশ: "হৃৎপিণ্ড একটি পেশিবহুল অঙ্গ। এটি বক্ষগহ্বরে অবস্থিত। হৃৎপিণ্ডের প্রধান কাজ সারা দেহে রক্ত সঞ্চালন করা। হৃৎপিণ্ড চারটি প্রকোষ্ঠে বিভক্ত।"
উত্তর: {{"triples": [{{"subject": "হৃৎপিণ্ড", "relation": "অবস্থান", "object": "বক্ষগহ্বর"}}, {{"subject": "হৃৎপিণ্ড", "relation": "কাজ", "object": "সারা দেহে রক্ত সঞ্চালন"}}, {{"subject": "হৃৎপিণ্ড", "relation": "অংশ", "object": "চারটি প্রকোষ্ঠ"}}]}}

উদাহরণ ২ (না-বাচক ও শিরোনাম — কিছুই নেবে না)
পাঠ্যাংশ: "৬.২ পানি ও খনিজ লবণ শোষণ\nঅনেক সময় উচ্চ রক্তচাপের কোনো লক্ষণ প্রকাশ পায় না।"
উত্তর: {{"triples": []}}

শুধু এই ফরম্যাটে JSON দাও, অন্য কোনো লেখা নয়:
{{"triples": [{{"subject": "...", "relation": "...", "object": "..."}}]}}

পাঠ্যাংশ:
\"\"\"{chunk}\"\"\""""


def build(chunk):
    return tok.apply_chat_template(
        [{"role": "system", "content": SYSTEM},
         {"role": "user", "content": PROMPT.format(chunk=chunk,
                                                   relations="、".join(RELATIONS))}],
        tokenize=False, add_generation_prompt=True)


def extract_json(text):
    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M).strip()
    start = text.find("{")
    if start < 0:
        return None
    depth, in_str, esc = 0, False, False
    for i, c in enumerate(text[start:], start):
        if esc:
            esc = False
        elif c == "\\":
            esc = True
        elif c == '"':
            in_str = not in_str
        elif not in_str:
            depth += (c == "{") - (c == "}")
            if depth == 0:
                try:
                    return json.loads(text[start:i + 1])
                except json.JSONDecodeError:
                    return None
    return None

In [ ]:
done = set()
if TRIPLES_JL.exists():
    with open(TRIPLES_JL, encoding="utf-8") as f:
        done = {json.loads(l)["chunk_id"] for l in f if l.strip()}
    print(f"resuming — {len(done)} chunks already done")

pending = todo[~todo.chunk_id.isin(done)].reset_index(drop=True)
print(f"{len(pending)} to go")


@torch.inference_mode()
def run(batch_df):
    enc = tok([build(t) for t in batch_df.text],
              return_tensors="pt", padding=True).to(model.device)
    gen = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                         pad_token_id=tok.pad_token_id)
    return [tok.decode(g[enc.input_ids.shape[1]:], skip_special_tokens=True) for g in gen]


t0 = time.time()
with open(TRIPLES_JL, "a", encoding="utf-8") as sink:
    for i in range(0, len(pending), BATCH):
        part = pending.iloc[i:i + BATCH]
        for row, raw in zip(part.itertuples(), run(part)):
            obj = extract_json(raw)
            sink.write(json.dumps({
                "chunk_id": row.chunk_id, "chapter_no": int(row.chapter_no),
                "ok": bool(obj and "triples" in obj),
                "triples": (obj or {}).get("triples", []), "raw": raw,
            }, ensure_ascii=False) + "\n")
        sink.flush()
        el = time.time() - t0
        n = min(i + BATCH, len(pending))
        print(f"  {n}/{len(pending)}  {el:.0f}s  ({el/max(n,1):.1f}s/chunk)", end="\r")
print(f"\ndone in {time.time()-t0:.0f}s")

## Stage 5 — collect

In [ ]:
# The model will not reliably obey the relation enum stated in the prompt, and it turns
# negated sentences ("লক্ষণ প্রকাশ পায় না") into positive assertions. Both are corrected
# here deterministically rather than hoped away in the prompt: a relation that contains an
# allowed one is snapped to it, anything else is dropped, and negated objects are refused.
# না must be matched as a standalone particle — ঘটনা, বেদনা, চেতনা, রচনা all end in না.
NEGATION = re.compile(r"(?:^|\s)(না|নেই|নয়|নাই)\s*$")
PRONOUNS = {"তুমি", "তোমরা", "আমরা", "আমি", "সে", "তারা", "তোমার", "আমাদের"}


def snap(rel):
    rel = rel.strip()
    if rel in RELATIONS:
        return rel
    for r in RELATIONS:
        if r in rel:
            return r
    return None


def validate(subj, rel, obj):
    subj, obj = subj.strip(), obj.strip()
    r = snap(rel)
    if r is None:
        return None, f"off-vocab relation: {rel}"
    if subj in PRONOUNS:
        return None, f"pronoun subject: {subj}"
    if NEGATION.search(obj):
        return None, f"negated object: {obj}"
    if len(subj) < 2 or len(obj) < 2:
        return None, "too short"
    return {"subject": subj, "relation": r, "object": obj}, None


records = [json.loads(l) for l in open(TRIPLES_JL, encoding="utf-8") if l.strip()]
rows, dropped, unparsed = [], [], 0
for rec in records:
    if not rec["ok"]:
        unparsed += 1
        continue
    for t in rec["triples"]:
        if not (isinstance(t, dict) and {"subject", "relation", "object"} <= t.keys()):
            continue
        good, why = validate(str(t["subject"]), str(t["relation"]), str(t["object"]))
        if good:
            rows.append({"chunk_id": rec["chunk_id"], "chapter_no": rec["chapter_no"], **good})
        else:
            dropped.append((why, t))

triples = pd.DataFrame(rows)
triples.to_csv(TRIPLES_CSV, index=False)

total = len(rows) + len(dropped)
print(f"{len(records)} chunks, {unparsed} unparseable")
print(f"{total} raw triples -> {len(rows)} kept, {len(dropped)} dropped "
      f"({len(rows)/total*100:.0f}% retained)" if total else "no triples")
reasons = collections.Counter(w.split(":")[0] for w, _ in dropped)
for why, n in reasons.most_common():
    print(f"   {n:>4}  {why}")
if len(triples):
    print()
    print(triples.relation.value_counts().to_string())
    assert triples.relation.isin(RELATIONS).all(), "off-vocabulary relation survived"
triples.head(20)
